# Machine Learning Pipeline Implementation
## Advanced Credit Risk Prediction System

This notebook demonstrates a complete ML pipeline using scikit-learn, TensorFlow, and PyTorch for credit risk assessment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 1. Data Loading and Initial Exploration

In [ ]:
from sklearn.datasets import fetch_openml

data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
df = data.frame

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df['class'].value_counts())

## 2. Data Preprocessing Pipeline

In [ ]:
df_processed = df.copy()

label_encoders = {}
for column in df_processed.columns:
    if df_processed[column].dtype == 'object' or df_processed[column].dtype == 'category':
        le = LabelEncoder()
        df_processed[column] = le.fit_transform(df_processed[column].astype(str))
        label_encoders[column] = le

X = df_processed.drop('class', axis=1)
y = df_processed['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=df_processed, x='duration', hue='class', kde=True)
plt.title('Duration Distribution by Class')

plt.subplot(1, 3, 2)
sns.histplot(data=df_processed, x='credit_amount', hue='class', kde=True)
plt.title('Credit Amount Distribution by Class')

plt.subplot(1, 3, 3)
correlation_matrix = df_processed.corr()
sns.heatmap(correlation_matrix[['class']].sort_values(by='class', ascending=False).head(10), 
            annot=True, cmap='coolwarm', center=0)
plt.title('Top 10 Features Correlation with Target')

plt.tight_layout()
plt.show()

## 3. Model Training - Scikit-learn Framework

In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Random Forest Performance:")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.4f}")

In [ ]:
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)

gb_pred = gb_model.predict(X_test_scaled)
gb_pred_proba = gb_model.predict_proba(X_test_scaled)[:, 1]

print("Gradient Boosting Performance:")
print(classification_report(y_test, gb_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, gb_pred_proba):.4f}")

## 4. Hyperparameter Tuning - GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test_scaled)
best_rf_pred_proba = best_rf.predict_proba(X_test_scaled)[:, 1]

print("\nTuned Random Forest Performance:")
print(classification_report(y_test, best_rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, best_rf_pred_proba):.4f}")

## 5. TensorFlow/Keras Deep Learning Model

In [ ]:
tf_model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

history = tf_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

tf_pred_proba = tf_model.predict(X_test_scaled).flatten()
tf_pred = (tf_pred_proba > 0.5).astype(int)

print("TensorFlow Model Performance:")
print(classification_report(y_test, tf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, tf_pred_proba):.4f}")

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['auc'], label='Training AUC')
plt.plot(history.history['val_auc'], label='Validation AUC')
plt.title('Model AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.show()

## 6. PyTorch Deep Learning Model

In [ ]:
class CreditRiskNN(nn.Module):
    def __init__(self, input_size):
        super(CreditRiskNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.sigmoid(self.fc4(x))
        return x

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

pytorch_model = CreditRiskNN(X_train_scaled.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(pytorch_model.parameters(), lr=0.001, weight_decay=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 100
train_losses = []

for epoch in range(epochs):
    pytorch_model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = pytorch_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

pytorch_model.eval()
with torch.no_grad():
    pytorch_pred_proba = pytorch_model(X_test_tensor).numpy().flatten()
    pytorch_pred = (pytorch_pred_proba > 0.5).astype(int)

print("\nPyTorch Model Performance:")
print(classification_report(y_test, pytorch_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, pytorch_pred_proba):.4f}")

## 7. Model Comparison and Evaluation

In [ ]:
models_comparison = {
    'Random Forest': {'pred': rf_pred, 'proba': rf_pred_proba},
    'Gradient Boosting': {'pred': gb_pred, 'proba': gb_pred_proba},
    'Tuned RF': {'pred': best_rf_pred, 'proba': best_rf_pred_proba},
    'TensorFlow': {'pred': tf_pred, 'proba': tf_pred_proba},
    'PyTorch': {'pred': pytorch_pred, 'proba': pytorch_pred_proba}
}

results_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])

for model_name, predictions in models_comparison.items():
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    accuracy = accuracy_score(y_test, predictions['pred'])
    precision = precision_score(y_test, predictions['pred'])
    recall = recall_score(y_test, predictions['pred'])
    f1 = f1_score(y_test, predictions['pred'])
    roc_auc = roc_auc_score(y_test, predictions['proba'])
    
    results_df = pd.concat([results_df, pd.DataFrame([{
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    }])], ignore_index=True)

print("\nModel Comparison Summary:")
print(results_df.to_string(index=False))

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
results_df.plot(x='Model', y=['Accuracy', 'Precision', 'Recall', 'F1-Score'], kind='bar', ax=plt.gca())
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')

plt.subplot(2, 2, 2)
for model_name, predictions in models_comparison.items():
    fpr, tpr, _ = roc_curve(y_test, predictions['proba'])
    auc = roc_auc_score(y_test, predictions['proba'])
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')

plt.subplot(2, 2, 3)
cm = confusion_matrix(y_test, best_rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Best Model (Tuned RF)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.subplot(2, 2, 4)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')

plt.tight_layout()
plt.show()

## 8. Model Saving and Deployment Pipeline

In [ ]:
joblib.dump(best_rf, 'best_random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

tf_model.save('tensorflow_model.h5')

torch.save(pytorch_model.state_dict(), 'pytorch_model.pth')

print("Models saved successfully!")
print("\nSaved files:")
print("1. best_random_forest_model.pkl - Tuned Random Forest model")
print("2. scaler.pkl - Feature scaler")
print("3. label_encoders.pkl - Categorical encoders")
print("4. tensorflow_model.h5 - TensorFlow/Keras model")
print("5. pytorch_model.pth - PyTorch model weights")

In [ ]:
loaded_rf_model = joblib.load('best_random_forest_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

sample_prediction = loaded_rf_model.predict(X_test_scaled[:5])
sample_proba = loaded_rf_model.predict_proba(X_test_scaled[:5])

print("Sample predictions from loaded model:")
for i in range(5):
    print(f"Sample {i+1}: Prediction = {sample_prediction[i]}, Probability = {sample_proba[i]}")

print("\nModel loading and inference successful!")

## 9. Cross-Validation Analysis

In [ ]:
from sklearn.model_selection import cross_validate

cv_scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = cross_validate(
    best_rf,
    X_train_scaled,
    y_train,
    cv=5,
    scoring=cv_scoring,
    return_train_score=True
)

print("Cross-Validation Results (5-Fold):")
print("="*60)
for metric in cv_scoring:
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    print(f"{metric.upper()}:")
    print(f"  Training: {train_scores.mean():.4f} (+/- {train_scores.std() * 2:.4f})")
    print(f"  Validation: {test_scores.mean():.4f} (+/- {test_scores.std() * 2:.4f})")
    print()

## 10. Pipeline Summary and Recommendations

In [ ]:
best_model_idx = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_auc = results_df.loc[best_model_idx, 'ROC-AUC']

print("Pipeline Execution Summary")
print("="*80)
print(f"Dataset: German Credit Risk (1000 samples, {X.shape[1]} features)")
print(f"Train/Test Split: 80/20")
print(f"\nModels Evaluated: {len(models_comparison)}")
print(f"  - Scikit-learn: Random Forest, Gradient Boosting, Tuned RF")
print(f"  - TensorFlow/Keras: Deep Neural Network")
print(f"  - PyTorch: Custom Neural Network")
print(f"\nBest Performing Model: {best_model_name}")
print(f"Best ROC-AUC Score: {best_auc:.4f}")
print(f"\nHyperparameter Tuning: GridSearchCV with 5-fold cross-validation")
print(f"Model Persistence: All models saved for deployment")
print("="*80)